# Предобработка ветвей для T и P: визуальная проверка

Ноутбук показывает результаты `run_branch_preprocessing.py` на одном нейроне:

1. примеры **толстых** и **тонких** ветвей, каждая на своём 3D-графике: сетка ветви, исходный скелет, восстановленная центральная линия, центры сечений (зелёные — достоверные, красные — с флагом; при наведении видны флаги), положения шипиков;
2. **непригодные для обучения** ветви и **причины**: для каждой ветви выводится, какое значение превысило какой порог;
3. ветви без центральной линии (`no_centerline`) и слишком короткие (`too_short`).

Только чтение: ничего не пересчитывается. Сетки ветвей читаются с хранилища (по одной ветви), поэтому для больших ветвей график строится несколько секунд.


In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neuron_model").is_dir():
            return candidate
    raise RuntimeError("Repository root was not found.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    pd.set_option("future.no_silent_downcasting", True)
except Exception:
    pass

from src.neuron_model import branch_geometry as bg
from src.neuron_model.branch_preprocessing import load_config

# --- параметры (можно задать переменными окружения) -------------------------------------------------------------
BRANCHES_ROOT = Path(os.environ.get("BRANCHES_ROOT", r"C:/Users/Student/LipsEkaterina/data/minnie65/preprocessed_branches"))
RAW_ROOT = Path(os.environ.get("RAW_ROOT", r"O:/Datasets/minnie65"))   # папка с нейронами: <RAW_ROOT>/<neuron_id>/limb_xxx/branch_xxx/branch_mesh.off
NEURON_ID = os.environ.get("NEURON_ID", "864691134886335738")
CONFIG_PATH = REPO_ROOT / "data/processed/config/branch_preprocessing.yaml"   # текущие пороги пригодности
N_EXAMPLES = 2          # сколько примеров каждого вида рисовать
SHOW_MESH = True        # сетка ветви (с хранилища); False - только линии и точки
THIN_NM, THICK_NM = 150.0, 200.0   # тонкая: медианный радиус < THIN_NM; толстая: >= THICK_NM

cfg = load_config(CONFIG_PATH)
print("корень результатов:", BRANCHES_ROOT)
print("пороги пригодности:", {k: getattr(cfg, k) for k in ("invalid_section_fraction_max", "spline_rms_factor", "spline_rms_floor_nm", "long_gap_nm", "trim_max_nm", "length_min_t_nm", "unknown_fraction_max", "position_fraction_min")})
print("пороги геометрии:", {k: getattr(cfg, k) for k in ("kink_factor", "kink_floor_nm", "large_radius_factor", "position_outlier_factor", "position_outlier_floor_nm", "min_section_points", "end_distance_factor", "min_reliable_sections")})


In [ ]:
branches = pd.read_parquet(BRANCHES_ROOT / NEURON_ID / "branches.parquet")
spines_all = pd.read_parquet(BRANCHES_ROOT / NEURON_ID / "branch_spines.parquet")
print(f"{len(branches)} объединённых ветвей нейрона {NEURON_ID}")
display(branches["status"].value_counts().rename("ветвей").to_frame())
ok = branches[branches["status"] == "ok"]
print("статус ok: годны для T —", int(ok["train_eligible_T"].sum()), ", для P —", int(ok["train_eligible_P"].sum()), "из", len(ok))
display(pd.crosstab(pd.cut(ok["radius_median_nm"], [0, 120, 150, 200, 1e9], labels=["< 120 нм", "120-150 нм", "150-200 нм", ">= 200 нм"]), ok["train_eligible_T"].map({True: "годна для T", False: "не годна для T"})))


## Вспомогательные функции

`explain(row)` перечисляет, какие проверки ветвь не прошла и по какому значению (сравнение с порогами из `branch_preprocessing.yaml`). Флаги в таблице записаны на момент прогона, а пороги пригодности можно потом менять (`reapply_eligibility.py`), поэтому причины здесь пересчитаны по записанным величинам и **текущим** порогам.

`plot_branch(row)` строит график ветви. Слои можно скрывать щелчком по легенде; скрытый по умолчанию слой — «трубка по радиусу» (круглое сечение, построенное по центральной линии и радиусу: её отклонение от настоящей сетки и есть то, что проверяет обратная реконструкция).


In [ ]:
FLAG_NAMES = {1: "few_points", 2: "large_radius", 4: "kink", 8: "outside_mesh", 16: "end_section"}


def explain(row) -> str:
    """Причины непригодности ветви: значение > порог."""
    def missing(v):
        return v is None or (isinstance(v, float) and np.isnan(v))

    r = lambda k, d=np.nan: d if missing(row.get(k)) else row.get(k)
    flag = lambda k, d=False: d if missing(row.get(k)) else bool(row.get(k))
    out = []
    if row["status"] == "too_short":
        return f"too_short: длина по таблице {r('table_length_nm'):.0f} нм < {cfg.length_min_p_nm:.0f} нм (геометрия не считалась)"
    if row["status"] == "no_centerline":
        return (f"no_centerline: надёжных сечений {int(r('n_reliable_sections', 0))} из {int(r('n_sections', 0))} < {cfg.min_reliable_sections} "
                f"(длина по таблице {r('table_length_nm'):.0f} нм)")
    if row["status"] != "ok":
        return f"{row['status']}: {row.get('error')}"
    rmed = r("radius_median_nm")
    T, P = [], []
    L = r("length_nm")
    if not (cfg.length_min_t_nm <= L <= cfg.length_max_t_nm):
        T.append(f"длина {L:.0f} нм вне [{cfg.length_min_t_nm:.0f}; {cfg.length_max_t_nm:.0f}]")
    if r("merge_max_gap_nm", 0.0) > cfg.join_distance_nm or not flag("skeleton_topology_ok", True):
        T.append(f"merge: разрыв между членами {r('merge_max_gap_nm', 0):.0f} нм (порог {cfg.join_distance_nm:.0f}) / топология скелета ok: {flag('skeleton_topology_ok', True)}")
    if r("section_invalid_fraction", 1.0) > cfg.invalid_section_fraction_max:
        T.append(f"cross_section: недостоверных сечений {r('section_invalid_fraction', 1.0):.2f} > {cfg.invalid_section_fraction_max}")
    lim = max(cfg.spline_rms_factor * rmed, cfg.spline_rms_floor_nm)
    gap_bad = r("longest_gap_nm", np.inf) > cfg.long_gap_nm
    rms_bad = r("centerline_rms_to_centers_nm", np.inf) > lim
    if gap_bad or rms_bad:
        s_ = []
        if rms_bad:
            s_.append(f"отклонение линии от центров {r('centerline_rms_to_centers_nm'):.0f} нм > {lim:.0f} нм (= max({cfg.spline_rms_factor}·r_med {rmed:.0f}, пол {cfg.spline_rms_floor_nm:.0f}))")
        if gap_bad:
            s_.append(f"наибольший пропуск линии {r('longest_gap_nm'):.0f} нм > {cfg.long_gap_nm:.0f}")
        T.append("centerline: " + "; ".join(s_))
        P.append("centerline")
    if r("twist_max_per_nm", np.inf) > cfg.twist_tolerance_per_nm:
        T.append(f"frame: кручение {r('twist_max_per_nm'):.1e} > {cfg.twist_tolerance_per_nm:.0e}")
    if flag("tube_self_contact", True) or r("max_r_kappa", np.inf) >= 1.0:
        T.append(f"tube: самоконтакт {flag('tube_self_contact', True)}, max r·κ {r('max_r_kappa', np.inf):.2f}")
    if not (cfg.radius_min_nm <= rmed <= cfg.radius_max_nm):
        T.append(f"радиус {rmed:.0f} нм вне [{cfg.radius_min_nm:.0f}; {cfg.radius_max_nm:.0f}]"); P.append("радиус")
    if r("trim_start_nm", np.inf) > cfg.trim_max_nm or r("trim_end_nm", np.inf) > cfg.trim_max_nm:
        T.append(f"trim: обрезка концов {r('trim_start_nm'):.0f} / {r('trim_end_nm'):.0f} нм > {cfg.trim_max_nm:.0f}")
    if missing(row.get("soma_distance_start_nm")) or flag("soma_distance_mismatch"):
        T.append("soma: расстояние до сомы не определено или расходится с таблицей")
    in_range = r("n_spines_in_range", 0)
    if r("unknown_fraction", 1.0) > cfg.unknown_fraction_max:
        P.append(f"spines: сегментов без вердикта {r('unknown_fraction', 1.0):.2f} > {cfg.unknown_fraction_max}")
    if in_range and r("n_with_position", 0) < cfg.position_fraction_min * in_range:
        P.append(f"spines: положение определено у {int(r('n_with_position', 0))} из {int(in_range)} шипиков (нужно >= {cfg.position_fraction_min:.0%})")
    if flag("train_eligible_T") and flag("train_eligible_P"):
        return "годна для T и P"
    note = " (флаги в таблице записаны при другом наборе порогов: выполните reapply_eligibility.py)"
    parts = []
    if not flag("train_eligible_T"):
        parts.append("T: " + " | ".join(T) if T else "T: по текущим порогам причин нет" + note)
    if not flag("train_eligible_P"):
        parts.append("P: " + " | ".join(P) if P else "P: по текущим порогам причин нет" + note)
    return "  ||  ".join(parts)


def load_geometry(row):
    path = BRANCHES_ROOT / NEURON_ID / f"limb_{int(row['limb_id']):03d}" / f"merged_{int(row['merged_branch_id'])}"
    out = {"sections": None, "geometry": None}
    if (path / "branch_sections.npz").exists():
        out["sections"] = dict(np.load(path / "branch_sections.npz"))
    if (path / "branch_geometry.npz").exists():
        out["geometry"] = dict(np.load(path / "branch_geometry.npz"))
    return out


def load_mesh(row):
    import trimesh

    members = [int(x) for x in str(row.get("member_branch_ids", "")).split("|") if x]
    vs, fs, offset = [], [], 0
    for b in members:
        p = RAW_ROOT / NEURON_ID / f"limb_{int(row['limb_id']):03d}" / f"branch_{b:03d}" / "branch_mesh.off"
        m = trimesh.load(p, process=False)
        vs.append(np.asarray(m.vertices, float)); fs.append(np.asarray(m.faces) + offset); offset += len(m.vertices)
    return (np.vstack(vs), np.vstack(fs)) if vs else (None, None)


def folders(row):
    """Имена папок в данных: limb_001/branch_10035 (+ другие члены объединённой ветви)."""
    members = [int(x) for x in str(row.get("member_branch_ids", "")).split("|") if x]
    limb = f"limb_{int(row['limb_id']):03d}"
    names = ", ".join(f"branch_{b:03d}" for b in members) if members else f"branch_{int(row['merged_branch_id']):03d}"
    return f"{limb}/{names}  (объединённая ветвь {int(row['merged_branch_id'])}" + (f", членов: {len(members)})" if len(members) > 1 else ")")


def spine_counts(row):
    sp = spines_all[(spines_all["limb_id"] == row["limb_id"]) & (spines_all["merged_branch_id"] == row["merged_branch_id"])]
    if not len(sp):
        return "сегментов шипиков в папках нет"
    st = sp["status"].value_counts().to_dict()
    pos = int(sp["has_position"].fillna(False).sum())
    return (f"сегментов в папках spines/: {len(sp)} (настоящих шипиков {st.get('valid', 0)}, ложных — части ствола {st.get('invalid', 0)}, без вердикта {st.get('unknown', 0)}); "
            f"с определённым положением: {pos}")


def section_flag_counts(row):
    n = row.get("n_sections")
    if n is None or (isinstance(n, float) and np.isnan(n)):
        return ""
    parts = [f"{name} {int(row.get('sections_' + name) or 0)}" for name in FLAG_NAMES.values() if f"sections_{name}" in row]
    return f"сечений {int(n)}, достоверных {int(row.get('n_reliable_sections') or 0)}; флаги (сечение может нести несколько): " + ", ".join(parts)


def flag_text(f):
    names = [n for bit, n in FLAG_NAMES.items() if int(f) & bit]
    return ", ".join(names) if names else "достоверное"


def plot_branch(row, show_mesh=SHOW_MESH, title_extra=""):
    g = load_geometry(row)
    sec, geo = g["sections"], g["geometry"]
    fig = go.Figure()
    if show_mesh:
        try:
            v, f = load_mesh(row)
            fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2], color="lightgrey", opacity=0.25, name="сетка ветви (с шипиками)", flatshading=True))
        except Exception as exc:
            print("сетка недоступна:", repr(exc)[:150])
    if geo is not None:
        sk = geo["skeleton"]
        fig.add_trace(go.Scatter3d(x=sk[:, 0], y=sk[:, 1], z=sk[:, 2], mode="lines", line=dict(color="black", width=2, dash="dot"), name="исходный скелет (на мембране)"))
        gm = geo["gamma"]
        fig.add_trace(go.Scatter3d(x=gm[:, 0], y=gm[:, 1], z=gm[:, 2], mode="lines+markers", marker=dict(size=2), line=dict(color="royalblue", width=6), name="центральная линия (сплайн), узлы 500 нм"))
        v_, f_ = bg.tube_mesh(gm, geo["radius"], geo["frame_e1"], geo["frame_e2"], n_theta=24, caps=False)
        fig.add_trace(go.Mesh3d(x=v_[:, 0], y=v_[:, 1], z=v_[:, 2], i=f_[:, 0], j=f_[:, 1], k=f_[:, 2], color="royalblue", opacity=0.3, name="трубка по радиусу (круг)", visible="legendonly"))
        if "ellipse_a" in geo:
            ve, fe = bg.ellipse_tube_mesh(gm, geo["ellipse_a"], geo["ellipse_b"], geo["ellipse_angle"], geo["frame_e1"], geo["frame_e2"], n_theta=24)
            fig.add_trace(go.Mesh3d(x=ve[:, 0], y=ve[:, 1], z=ve[:, 2], i=fe[:, 0], j=fe[:, 1], k=fe[:, 2], color="darkorange", opacity=0.3,
                                    name=f"трубка-эллипс (b/a медиана {np.median(geo['ellipse_b'] / geo['ellipse_a']):.2f})", visible="legendonly"))
    elif sec is not None:
        p = sec["points"]
        fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="lines", line=dict(color="black", width=2, dash="dot"), name="исходный скелет (перепробован)"))
    if sec is not None:
        c, fl, rad = sec["centers"], sec["flags"], sec["radii"]
        fit = np.isfinite(rad)
        good = fit & (fl == 0)
        bad = fit & (fl != 0)
        fig.add_trace(go.Scatter3d(x=c[good, 0], y=c[good, 1], z=c[good, 2], mode="markers", marker=dict(size=4, color="green"), name=f"центры сечений: достоверные ({int(good.sum())})",
                                   text=[f"r {r_:.0f} нм" for r_ in rad[good]], hoverinfo="text"))
        fig.add_trace(go.Scatter3d(x=c[bad, 0], y=c[bad, 1], z=c[bad, 2], mode="markers", marker=dict(size=5, color="red", symbol="diamond"), name=f"центры сечений: с флагом ({int(bad.sum())})",
                                   text=[f"{flag_text(f_)}; r {r_:.0f} нм" for f_, r_ in zip(fl[bad], rad[bad])], hoverinfo="text"))
    if geo is not None and "s" in spines_all.columns:  # no spine of the neuron has a position -> no column s
        sp = spines_all[(spines_all["limb_id"] == row["limb_id"]) & (spines_all["merged_branch_id"] == row["merged_branch_id"]) & (spines_all["status"] == "valid") & spines_all["s"].notna()]
        if len(sp):
            s_, th = sp["s"].to_numpy(float), sp["theta"].to_numpy(float)
            interp = lambda a: np.stack([np.interp(s_, geo["s"], a[:, k]) for k in range(3)], 1)
            r_ = np.interp(s_, geo["s"], geo["radius"])
            anchor = interp(geo["gamma"]) + r_[:, None] * (np.cos(th)[:, None] * interp(geo["frame_e1"]) + np.sin(th)[:, None] * interp(geo["frame_e2"]))
            kind = np.where(sp["projects_to_end"].fillna(False), "за концом линии", np.where(sp["position_outlier"].fillna(False), "выброс по расстоянию", "положение определено"))
            for name, color in (("положение определено", "magenta"), ("выброс по расстоянию", "orange"), ("за концом линии", "red")):
                m = kind == name
                if m.any():
                    fig.add_trace(go.Scatter3d(x=anchor[m, 0], y=anchor[m, 1], z=anchor[m, 2], mode="markers", marker=dict(size=5, color=color, symbol="circle"), name=f"основания шипиков: {name} ({int(m.sum())})",
                                               text=[f"u {u:.2f}, θ {t:.2f}, |d-r| {abs(d):.0f} нм" for u, t, d in zip(sp["u"][m], sp["theta"][m], sp["dist_perp_minus_radius"][m])], hoverinfo="text"))
    head = f"{folders(row)}: длина {row.get('length_nm', row.get('table_length_nm', np.nan)) / 1000:.1f} мкм, r_med {row.get('radius_median_nm', np.nan):.0f} нм"
    sub = spine_counts(row)
    fig.update_layout(title=dict(text=head + "<br><sup>" + sub + (" | " + title_extra if title_extra else "") + "</sup>", font=dict(size=13)), scene=dict(aspectmode="data"), height=720, legend=dict(font=dict(size=10)))
    return fig


def show_examples(frame, n, header):
    print(header)
    for _, row in frame.head(n).iterrows():
        print(f"\n{folders(row)}:", explain(row.to_dict()))
        print("   ", section_flag_counts(row.to_dict()))
        plot_branch(row.to_dict()).show()


## 1. Толстые ветви (медианный радиус ≥ 200 нм), годные для обучения T

На графике: линия должна идти по оси сетки; почти все центры сечений — зелёные; основания шипиков (пурпурные) лежат на поверхности ствола.

In [ ]:
thick = ok[(ok["radius_median_nm"] >= THICK_NM) & ok["train_eligible_T"]].sort_values("length_nm", ascending=False)
show_examples(thick, N_EXAMPLES, f"Толстые, годные для T: {len(thick)} ветвей; показаны самые длинные")


## 2. Тонкие ветви (медианный радиус < 150 нм)

Тонкие ветви с шумом центров порядка радиуса: здесь важно, как выглядят красные центры и где линия расходится с сеткой. Сначала годные (если есть), потом отбракованные.

In [ ]:
thin = ok[ok["radius_median_nm"] < THIN_NM].sort_values("length_nm", ascending=False)
thin_good = thin[thin["train_eligible_T"]]
thin_bad = thin[~thin["train_eligible_T"]]
print(f"Тонкие ветви: {len(thin)}, из них годных для T: {len(thin_good)}")
show_examples(thin_good, N_EXAMPLES, "Тонкие, годные для T")
show_examples(thin_bad, N_EXAMPLES, "Тонкие, НЕ годные для T (самые длинные)")


## 3. Непригодные для обучения ветви: причины

Таблица и диаграмма считают, **по каким проверкам** ветви не прошли отбор (одна ветвь может не пройти несколько). Проверка длины для T — отдельная строка.

In [ ]:
from collections import Counter

bad_T = ok[~ok["train_eligible_T"]]
bad_P = ok[~ok["train_eligible_P"]]
cT, cP = Counter(), Counter()
for _, row in bad_T.iterrows():
    for part in str(row["qc_reason"] or "").split(";"):
        if part:
            cT[part] += 1
    if row["length_nm"] < cfg.length_min_t_nm:
        cT["длина < 5 мкм"] += 1
for _, row in bad_P.iterrows():
    for part in str(row["qc_reason"] or "").split(";"):
        if part:
            cP[part] += 1
fig = go.Figure()
fig.add_bar(x=list(cT.keys()), y=list(cT.values()), name=f"не годны для T ({len(bad_T)} из {len(ok)})")
fig.add_bar(x=list(cP.keys()), y=list(cP.values()), name=f"не годны для P ({len(bad_P)} из {len(ok)})")
fig.update_layout(title="Проверки, не пройденные ветвями со статусом ok (флаги записаны при прогоне)", barmode="group", height=420)
fig.show()
bad_table = bad_T.assign(причины=[explain(r.to_dict()) for _, r in bad_T.iterrows()])[["limb_id", "merged_branch_id", "length_nm", "radius_median_nm", "section_invalid_fraction", "centerline_rms_to_centers_nm", "причины"]]
pd.set_option("display.max_colwidth", 200)
display(bad_table.sort_values("length_nm", ascending=False).head(25))


## 4. Примеры непригодных ветвей по основным причинам

Для каждой из наиболее частых причин показаны ветви, которые не проходят **только** по ней (если такие есть) или самые длинные из не прошедших.

In [ ]:
REASONS = [r for r, _ in cT.most_common(3) if r != "длина < 5 мкм"]
for reason in REASONS:
    only = bad_T[bad_T["qc_reason"].fillna("") == reason]
    pool = only if len(only) else bad_T[bad_T["qc_reason"].fillna("").str.contains(reason)]
    show_examples(pool.sort_values("length_nm", ascending=False), 1, f"== причина: {reason}; ветвей {len(pool)} (только по ней: {len(only)}) ==")


## 5. Ветви без центральной линии и слишком короткие

`no_centerline` — меньше 5 достоверных сечений (линия строится по сечениям с шагом 500 нм; у коротких ветвей концевые участки 2·r_med у каждого конца считаются ненадёжными). `too_short` — короче 2 мкм, геометрия не считалась.

In [ ]:
nc = branches[branches["status"] == "no_centerline"].sort_values("table_length_nm", ascending=False)
ts = branches[branches["status"] == "too_short"]
print(f"no_centerline: {len(nc)} ветвей, too_short: {len(ts)}")
if len(nc):
    display(nc[["limb_id", "merged_branch_id", "table_length_nm", "radius_median_nm", "n_sections", "n_reliable_sections", "member_count"]].head(15))
show_examples(nc, N_EXAMPLES, "no_centerline: самые длинные")
if len(ts):
    print("too_short: длины по таблице, нм:", ts["table_length_nm"].round().astype(int).tolist()[:30])
